# 音のプログラミング 第5回: オーディオエフェクトとダイナミクス

**前回まで学んだこと：**
- 基本的な音響合成（発振器、エンベロープ）
- フィルターによる音色デザイン
- 楽器音の作成

**今回の学習目標:**
- **リバーブ**（残響）で音に空間的な広がりを与える
- **ディレイ**（遅延）でエコー効果を作る
- **コーラス**で音を豊かにする
- **ダイナミクス**（音量制御）の基本

**所要時間:** 90分

## 🛠️ 環境セットアップ

In [ ]:
# 🛠️ 環境セットアップ
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
    print("🔧 Google Colab環境で実行中...")
except ImportError:
    IN_COLAB = False
    print("🏠 ローカル環境で実行中")

if IN_COLAB:
    print("🔧 Google Colab環境を設定中...")
    !pip install japanize-matplotlib
    !git clone https://github.com/ggszk/simple-audio-programming.git
    sys.path.append('/content/simple-audio-programming')
    import japanize_matplotlib
else:
    print("🔧 ローカル環境を設定中...")
    sys.path.append('..')
    import platform
    if platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'Hiragino Sans'
    else:
        plt.rcParams['font.family'] = 'Meiryo'

print("✅ セットアップ完了")

## 🛠️ 今回使用するライブラリ

In [ ]:
from audio_lib import (
    sine_wave, sawtooth_wave, adsr, AudioSignal,
    LowPassFilter,
    Reverb, Delay, Chorus, Compressor,
)
from audio_lib.synthesis.note_utils import note_to_frequency, note_name_to_number
from audio_lib.notebook import play_sound

## 🎵 準備: ピアノ音の作成関数

前回学んだ技法で、エフェクトのテスト用ピアノ音を作ります。

In [ ]:
def create_simple_piano(note_freq, duration=2.0):
    """シンプルなピアノ音"""
    signal = sine_wave(note_freq, duration)
    envelope = adsr(duration, attack=0.01, decay=0.3, sustain=0.4, release=1.5)
    return signal * envelope

## 🎯 実習1: リバーブ（残響）— 音に空間を与える

リバーブは部屋や空間での音の反響をシミュレートする効果です。

- **コンサートホール**：長い残響
- **小さな部屋**：短い残響
- **屋外**：ほぼ残響なし

In [ ]:
# ドライ音（リバーブなし）
dry_piano = create_simple_piano(note_to_frequency(note_name_to_number('C4')))
display(play_sound(dry_piano, "ドライ音（リバーブなし）"))

# リバーブありの音
reverb = Reverb(room_size=0.8, damping=0.5, wet_level=0.3)
wet_piano = reverb.process(dry_piano)
display(play_sound(wet_piano, "リバーブあり（大きな部屋）"))

### 聞き比べてみよう
- ドライ音：直接的で近い感じ
- リバーブあり：空間的で遠い感じ、音の尻尾が長い

### リバーブパラメータの効果

| パラメータ | 範囲 | 効果 |
|-----------|------|------|
| **room_size** | 0.1〜0.8 | 部屋の大きさ（小部屋〜大ホール） |
| **damping** | 0.1〜0.9 | 減衰率（低い=長く響く、高い=よく減衰） |
| **wet_level** | 0.1〜0.6 | エフェクト音の量（控えめ〜しっかり） |

In [ ]:
# 異なるリバーブ設定を比較
base_sound = create_simple_piano(note_to_frequency(note_name_to_number('A4')), 1.5)

settings = [
    ("小さな部屋", Reverb(room_size=0.3, damping=0.8, wet_level=0.2)),
    ("コンサートホール", Reverb(room_size=0.7, damping=0.4, wet_level=0.4)),
    ("大きなホール", Reverb(room_size=0.8, damping=0.3, wet_level=0.5)),
]

for name, reverb in settings:
    processed = reverb.process(base_sound)
    display(play_sound(processed, name))

## 🎯 実習2: ディレイ（遅延）— エコー効果を作る

ディレイは音を一定時間遅らせて元の音に重ねる効果です。

- **短いディレイ**：音の厚みや広がり
- **長いディレイ**：明確なエコー効果
- **フィードバック**：エコーの繰り返し

In [ ]:
base_sound = create_simple_piano(note_to_frequency(note_name_to_number('E4')), 1.0)

display(play_sound(base_sound, "元の音"))

# 短いディレイ（音の厚み）
delay_short = Delay(delay_time=0.1, feedback=0.3, wet_level=0.4)
display(play_sound(delay_short.process(base_sound), "短いディレイ（0.1秒）"))

# 長いディレイ（明確なエコー）
delay_long = Delay(delay_time=0.5, feedback=0.4, wet_level=0.5)
display(play_sound(delay_long.process(base_sound), "長いディレイ（0.5秒）"))

# フィードバック多め（エコーの繰り返し）
delay_feedback = Delay(delay_time=0.3, feedback=0.6, wet_level=0.4)
display(play_sound(delay_feedback.process(base_sound), "フィードバック多め（エコーの繰り返し）"))

## 🎯 実習3: コーラス — 音を豊かにする

コーラスは音を少しずつピッチや時間をずらしてコピーし、重ね合わせる効果です。

| パラメータ | 推奨範囲 | 効果 |
|-----------|---------|------|
| **depth** | 0.01〜0.03 | 変調の深さ（大きすぎると不安定に） |
| **rate** | 0.5〜5.0 Hz | 変調の速度 |
| **wet_level** | 0.2〜0.6 | エフェクト音の量 |

In [ ]:
base_sound = create_simple_piano(note_to_frequency(note_name_to_number('G4')), 2.0)

display(play_sound(base_sound, "元の音"))

# 軽いコーラス（自然な効果）
chorus_light = Chorus(rate=2.0, depth=0.015, wet_level=0.4)
display(play_sound(chorus_light.process(base_sound), "軽いコーラス"))

# 深いコーラス（しっかりとした効果）
chorus_deep = Chorus(rate=1.5, depth=0.025, wet_level=0.6)
display(play_sound(chorus_deep.process(base_sound), "深いコーラス"))

# 速いコーラス（ビブラート的）
chorus_fast = Chorus(rate=5.0, depth=0.01, wet_level=0.5)
display(play_sound(chorus_fast.process(base_sound), "速いコーラス（ビブラート的）"))

## 🎯 実習4: エフェクトの組み合わせ — プロフェッショナルな音作り

In [ ]:
def create_lush_piano(note_freq, duration=3.0):
    """豊かなエフェクトをかけたピアノ音"""
    base_sound = create_simple_piano(note_freq, duration)

    # エフェクトチェーン：コーラス → ディレイ → リバーブ
    chorus = Chorus(rate=2.5, depth=0.02, wet_level=0.3)
    delay = Delay(delay_time=0.15, feedback=0.25, wet_level=0.2)
    reverb = Reverb(room_size=0.6, wet_level=0.3)

    sound = chorus.process(base_sound)
    sound = delay.process(sound)
    sound = reverb.process(sound)
    return sound

dry_sound = create_simple_piano(note_to_frequency(note_name_to_number('C4')), 3.0)
lush_sound = create_lush_piano(note_to_frequency(note_name_to_number('C4')), 3.0)

display(play_sound(dry_sound, "ドライなピアノ音"))
display(play_sound(lush_sound, "豊かなエフェクト付きピアノ音"))

## 🎯 実習5: ダイナミクス処理の基本

ダイナミクスは音量の変化を制御する技術です。

- **コンプレッサー**：音量差を小さくする
- **リミッター**：音量の上限を設定
- **ゲート**：小さな音をカット

In [ ]:
# シンプルなコンプレッサー効果を実装
def simple_compressor(signal, threshold=0.7, ratio=4.0):
    """シンプルなコンプレッサー"""
    compressed = signal.data.copy()

    for i in range(len(compressed)):
        sample_level = abs(compressed[i])
        if sample_level > threshold:
            excess = sample_level - threshold
            compressed_excess = excess / ratio
            new_level = threshold + compressed_excess
            if compressed[i] >= 0:
                compressed[i] = new_level
            else:
                compressed[i] = -new_level

    return AudioSignal(compressed, signal.sample_rate)

# 動的な音量変化を持つ音を作成
def create_dynamic_sound(duration=4.0):
    """音量が変化する音を作成"""
    freq = note_to_frequency(note_name_to_number('A4'))
    signal = sine_wave(freq, duration)

    time_samples = len(signal)
    volume_envelope = np.ones(time_samples)
    for i in range(time_samples):
        t = i / time_samples
        volume = 0.5 + 0.3 * np.sin(t * 8 * np.pi) + 0.2 * np.sin(t * 20 * np.pi)
        volume_envelope[i] = max(0.1, min(1.0, volume))

    return AudioSignal(signal.data * volume_envelope, signal.sample_rate)

# ダイナミクス処理の比較
dynamic_sound = create_dynamic_sound()
compressed_sound = simple_compressor(dynamic_sound, threshold=0.6, ratio=3.0)

display(play_sound(dynamic_sound, "元の音（音量変化あり）"))
display(play_sound(compressed_sound, "コンプレッサー適用後（音量変化を抑制）"))

# 波形の比較
plt.figure(figsize=(12, 6))
time = np.linspace(0, 4, len(dynamic_sound))

plt.subplot(2, 1, 1)
plt.plot(time, dynamic_sound.data, alpha=0.7)
plt.title('元の音（音量変化あり）')
plt.ylabel('振幅')
plt.grid(True, alpha=0.3)

plt.subplot(2, 1, 2)
plt.plot(time, compressed_sound.data, alpha=0.7, color='orange')
plt.title('コンプレッサー適用後')
plt.xlabel('時間 (秒)')
plt.ylabel('振幅')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🎯 実習6: 音楽フレーズにエフェクトを適用

In [ ]:
# 音楽ジャンル別エフェクト設定
base_melody = [
    note_to_frequency(note_name_to_number('C4')),
    note_to_frequency(note_name_to_number('E4')),
    note_to_frequency(note_name_to_number('G4')),
    note_to_frequency(note_name_to_number('C5')),
]

def create_melody(notes, note_duration=1.0):
    """メロディを作成"""
    sounds = [create_simple_piano(freq, note_duration).data for freq in notes]
    return AudioSignal(np.concatenate(sounds), 44100)

base_sound = create_melody(base_melody, 1.5)

# クラシック：自然なリバーブ
reverb_classic = Reverb(room_size=0.8, wet_level=0.5)
display(play_sound(reverb_classic.process(base_sound), "クラシック（リバーブ重視）"))

# ポップス：コーラスで豊かさを
chorus = Chorus(rate=2.0, depth=0.015, wet_level=0.3)
delay = Delay(delay_time=0.125, feedback=0.2, wet_level=0.2)
pop_sound = delay.process(chorus.process(base_sound))
display(play_sound(pop_sound, "ポップス（コーラス + ディレイ）"))

# アンビエント：空間的な効果
reverb_ambient = Reverb(room_size=0.9, wet_level=0.7)
delay_ambient = Delay(delay_time=0.5, feedback=0.6, wet_level=0.4)
ambient_sound = delay_ambient.process(reverb_ambient.process(base_sound))
display(play_sound(ambient_sound, "アンビエント（深いリバーブ + ディレイ）"))

## 🏆 チャレンジ課題

### 課題1：自分だけのエフェクトプリセット

In [ ]:
def create_genre_preset(base_sound, genre="pop"):
    """ジャンル別エフェクトプリセット"""
    if genre == "pop":
        chorus = Chorus(rate=2.0, depth=0.015, wet_level=0.3)
        reverb = Reverb(room_size=0.5, damping=0.6, wet_level=0.2)
        sound = reverb.process(chorus.process(base_sound))
    elif genre == "ambient":
        delay = Delay(delay_time=0.6, feedback=0.5, wet_level=0.4)
        reverb = Reverb(room_size=0.9, damping=0.2, wet_level=0.6)
        chorus = Chorus(rate=1.0, depth=0.06, wet_level=0.4)
        sound = reverb.process(delay.process(chorus.process(base_sound)))
    elif genre == "dry":
        sound = simple_compressor(base_sound, threshold=0.8, ratio=2.0)
    else:
        sound = base_sound
    return sound

test_sound = create_simple_piano(note_to_frequency(note_name_to_number('A4')), 2.0)

for genre in ["dry", "pop", "ambient"]:
    processed = create_genre_preset(test_sound, genre)
    display(play_sound(processed, f"{genre.upper()}プリセット"))

### 課題2：エフェクトの順序による違い

In [ ]:
# エフェクトチェーンの順序比較
base_sound = create_simple_piano(note_to_frequency(note_name_to_number('C4')), 2.0)

reverb = Reverb(room_size=0.7, damping=0.4, wet_level=0.4)
delay = Delay(delay_time=0.3, feedback=0.3, wet_level=0.4)

# チェーン1：リバーブ → ディレイ
chain1 = delay.process(reverb.process(base_sound))
display(play_sound(chain1, "チェーン1（リバーブ → ディレイ）"))

# チェーン2：ディレイ → リバーブ
chain2 = reverb.process(delay.process(base_sound))
display(play_sound(chain2, "チェーン2（ディレイ → リバーブ）"))

聞き比べてみよう：
- チェーン1：エコーに残響がかかる
- チェーン2：残響にエコーがかかる

エフェクトの適用順序で音が大きく変わります。

### 課題3：ライブパフォーマンス風のエフェクト切り替え

In [ ]:
def create_performance_sound():
    """ライブパフォーマンス風：時間とともにエフェクトが変化"""
    sections = []
    base_freq = note_to_frequency(note_name_to_number('G4'))

    # セクション1：ドライ
    section1 = create_simple_piano(base_freq, 2.0)
    sections.append(section1.data)

    # セクション2：コーラス追加
    section2 = create_simple_piano(base_freq, 2.0)
    chorus = Chorus(rate=3.0, depth=0.015, wet_level=0.5)
    sections.append(chorus.process(section2).data)

    # セクション3：ディレイ追加
    section3 = create_simple_piano(base_freq, 2.0)
    delay = Delay(delay_time=0.25, feedback=0.4, wet_level=0.4)
    sections.append(delay.process(chorus.process(section3)).data)

    # セクション4：フルエフェクト
    section4 = create_simple_piano(base_freq, 2.0)
    reverb = Reverb(room_size=0.8, damping=0.3, wet_level=0.5)
    s4 = reverb.process(delay.process(chorus.process(section4)))
    sections.append(s4.data)

    return AudioSignal(np.concatenate(sections), 44100)

performance_sound = create_performance_sound()
display(play_sound(performance_sound,
    "パフォーマンス風（ドライ→コーラス→ディレイ→フル）"))

## 📚 今日のまとめ

### 学んだこと
1. **リバーブ**：空間的な広がりと深みを与える
2. **ディレイ**：エコー効果と音の厚みを作る
3. **コーラス**：音を豊かにし、複数楽器感を演出
4. **ダイナミクス処理**：音量変化の制御
5. **エフェクトチェーン**：複数エフェクトの組み合わせ
6. **ジャンル別プリセット**：用途に応じた設定

### 音楽制作のコツ
- エフェクトは適度に — やりすぎは禁物
- 順序が重要 — エフェクトの適用順で音が大きく変わる
- ジャンルに合わせる — 音楽スタイルに適した設定を選ぶ

### 次回予告
第6回では**MIDI**と**シーケンサー**を学びます。
複数パートを組み合わせた本格的な音楽制作に挑戦します。